In [ ]:
import Preprocessor
import Model

# Preprocessing

In [ ]:
data_preprocessor = Preprocessor.Preprocessor('historical_data/stockdata_imputed.db',
                                              path='database',
                                              drift=False,
                                              volatility=False,
                                              moving_avg=False,
                                              momentum=False,
                                              volume_features=False,
                                              support_resistance=False,
                                              bollinger_bands=False,
                                              z_score=False)

In [ ]:
# Create normalized db, store in 'historical_data/sd_pre_n.db'
normalized_data, X_sequences, y_sequences = data_preprocessor.normalize()

normalized_data.head()
X_sequences.shape()
y_sequences.shape()

# Create Model

In [ ]:
# Load your dataset
input_timesteps = 79
num_features = X_sequences.shape[2]  # Automatically infer number of features
max_output_horizon = 79

# Pass into any model
# PredictonModelObject = Model.StockLSTMModel(normalized_data, X_sequences, y_sequences, data_preprocessor.scalers)

# model = PredictonModelObject.create_variable_multistep_model(input_timesteps, num_features, max_output_horizon)

# model.fit(
#     X_sequences, y_sequences,
#     epochs=16,
#     batch_size=32,
#     validation_split=0.1
# )


In [ ]:
# Save the trained model to an H5 file
model.save("16epoch.h5")

# Predict on new stock

In [ ]:
# # Parameters for prediction
# prediction_date = '2023-06-01'  # Example date in your historical data
# known_timestamps = 50  # Number of known timestamps (out of 79 per day)
# horizon = 17  # Example horizon value (target stock price)

# # Call the predict function and plot results
# model.predict(normalized_data, data_preprocessor.scalers, prediction_date, known_timestamps, horizon)

In [ ]:
def predict_variable_steps(model, new_data, output_horizon):
    predictions = model.predict(new_data)
    return predictions[:, :output_horizon] 

# Load X timesteps of real data
new_ticker_close = [
    10.63, 10.615, 10.63, 10.59, 10.585, 10.594, 10.6091, 10.5866, 10.5599, 10.58,
    10.5459, 10.544, 10.57, 10.695, 10.685, 10.6883, 10.675, 10.69, 10.695, 10.655,
    10.69, 10.6798, 10.725, 10.695, 10.73, 10.735, 10.7299, 10.725, 10.6872, 10.695,
    10.7292, 10.6993, 10.705, 10.715, 10.685, 10.695, 10.655, 10.6149, 10.64, 10.665,
    10.653, 10.64, 10.7, 10.68, 10.68, 10.69, 10.6, 10.6, 10.66,
    10.65, 10.62, 10.65, 10.6, 10.62, 10.6299, 10.63, 10.63, 10.65,
    10.65, 10.63, 10.63, 10.67, 10.67
]

new_ticker_transactions = values = [
    1181, 702, 458, 456, 568, 433, 1035, 1167, 515, 720,
    564, 465, 794, 694, 720, 413, 550, 419, 477, 370,
    414, 436, 564, 653, 831, 470, 429, 404, 425, 644,
    410, 458, 519, 853, 388, 485, 558, 947, 1350, 1651,
    1859, 2674, 30, 46, 36, 48, 55, 28, 32, 76,
    13, 47, 8, 21, 4, 2, 2, 12, 7, 2,
    2, 23, 9
]

new_ticker_volumes = values = [
    187833, 173777, 70763, 55344, 76240, 61692, 266638, 247699, 110628, 119257,
    90344, 63151, 108678, 130077, 206748, 59165, 101482, 68758, 79811, 85783,
    65185, 118912, 139721, 159113, 143772, 72940, 77568, 66695, 68214, 129817,
    55880, 73040, 81163, 208149, 65722, 78427, 72962, 209455, 251227, 693261,
    336738, 562989, 3138, 7334, 6579, 4617, 8711, 4861, 3086, 13305,
    4612, 8016, 1146, 3364, 1101, 1101, 271, 4817, 238, 400,
    400, 1692, 2760
]




new_ticker_df = pd.DataFrame(zip(new_ticker_close, new_ticker_transactions, new_ticker_volumes),
                             columns=["close", "transactions", "volume"])

# Normalize using stored scalers from training
scaler_close = data_preprocessor.scalers["CHPT"]["close"]
scaler_transactions = data_preprocessor.scalers["CHPT"]["transactions"]
scaler_volume = data_preprocessor.scalers["CHPT"]["volume"]

new_ticker_df["close"] = scaler_close.transform(new_ticker_df[["close"]].to_numpy())
new_ticker_df["transactions"] = scaler_transactions.transform(new_ticker_df[["transactions"]].to_numpy())
new_ticker_df["volume"] = scaler_volume.transform(new_ticker_df[["volume"]].to_numpy())


# Ensure the shape is (1, 79, 3) by padding with zeros (or historical means)
needed_timesteps = 79 - len(new_ticker_df)
print(len(new_ticker_close))
print(len(new_ticker_transactions))
print(len(new_ticker_volumes))
print(len(new_ticker_df))
print("need:", needed_timesteps)
padding = np.zeros((needed_timesteps, 3))  # 3 features (close, transactions, volume)
padded_input = np.vstack([padding, new_ticker_df.to_numpy()])  # Stack to get (79,3)

# Reshape for model input (1, 79, 3)
model_input = padded_input.reshape(1, 79, 3)

# predict remaining 79-len rows
output_horizon = needed_timesteps  # remaining timesteps until end of day
predicted_prices_norm = predict_variable_steps(model, model_input, output_horizon)

# denormalize predictions to get real-world prices
predicted_prices = scaler_close.inverse_transform(predicted_prices_norm.reshape(-1, 1)).flatten()

# result information
print("Predicted Prices:", predicted_prices)
print("Output Shape:", predicted_prices.shape)  # expected: (79-len,)


In [ ]:
actual_forecast = [10.64,
10.64,
10.63,
10.64,
10.66,
10.66,
10.66,
10.7,
10.7,
10.68,
10.7,
10.69,
10.69,
10.6601,
10.6501,
10.6501]

In [ ]:
import matplotlib.pyplot as plt

# Generate timestamps from 9:00 AM to 4:00 PM in 5-minute intervals
timestamps = pd.date_range(start="09:00", end="16:00", periods=79).strftime('%H:%M')

# Create plot
plt.figure(figsize=(12, 6))

# Plot known data (first 30)
print("OI", len(timestamps[:pivot]))
plt.plot(timestamps[:pivot], new_ticker_close, color='orange', marker='o', label=f"Known (0 - {pivot})")

# Plot predicted data (31-79)
plt.plot(timestamps[pivot:79], predicted_prices, color='blue', linestyle='dashed', marker='o', label=f"Predicted ({pivot} - 79)")

# Assuming `actual_rest_of_day` contains actual price values for comparison

# Plot actual rest of the day data (31-79) in black
plt.plot(timestamps[pivot:79], actual, color='black', marker='o', label=f"Actual ({pivot}-79)")

# Formatting the plot
plt.xlabel("Time (Hour:Minute)")
plt.ylabel("Stock Price")
plt.title("Stock Price Prediction vs Actual")
plt.xticks(rotation=45)
plt.legend()
plt.grid()

# Show plot
plt.show()
